In [1]:
!pip install xgboost

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   - -------------------------------------- 3.9/101.7 MB 29.4 MB/s eta 0:00:04
   ----- ---------------------------------- 13.4/101.7 MB 40.0 MB/s eta 0:00:03
   -------- ------------------------------- 21.8/101.7 MB 40.5 MB/s eta 0:00:02
   ----------- ---------------------------- 30.4/101.7 MB 41.1 MB/s eta 0:00:02
   ------------- -------------------------- 33.8/101.7 MB 35.8 MB/s eta 0:00:02
   -------------- ------------------------- 37.7/101.7 MB 32.4 MB/s eta 0:00:02
   ---------------- ----------------------- 41.4/101.7 MB 29.9 MB/s eta 0:00:03
   ----------------- ---------------------- 45.4/101.7 MB 28.3 MB/s eta 0:00:02
   ------------------- -------------------- 48.5/101.7 MB 26.8 MB/s eta 0:00:02
   -------------------- ------------------- 52.7/101.7 MB 26.0 MB/s eta 0:00:02
   ---------------------- ----------------- 56.1/101


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [44]:
#import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import cross_val_score, KFold
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from xgboost import XGBRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, root_mean_squared_error

In [7]:
#load the dataset
df=pd.read_csv(r'C:\Users\boser\Desktop\DisasterProject\Dataset\df_cleaned_final.csv')
display(df.sample(10))

,Disaster Subgroup,Disaster Type,Disaster Subtype,Location,Magnitude,Magnitude Scale,Start Year,Start Month,Start Day,End Year,...,Primary_State,Deaths_log,Affected_log,Damage_log,Disaster Type_encoded,Disaster Subtype_encoded,Disaster Subgroup_encoded,Season_encoded,Primary_State_encoded,Magnitude Scale_encoded
372,Meteorological,Storm,Tropical cyclone,"Mandasa, Godalpur, Gajapati, Ganjam, Rayagada,...",126.00,Kph,2018,10.0,11.0,2018,...,Andhra Pradesh,4.454347,12.612208,13.732130,9,26,4,1,0,1
323,Meteorological,Storm,Lightning/Thunderstorms,Uttar Pradesh province,NaN,Kph,2001,5.0,24.0,2001,...,Uttar Pradesh,3.091042,10.308986,11.982935,9,16,4,2,26,1
331,Meteorological,Storm,Storm (General),"Barddhaman, North 24 Parganas, South 24 Pargan...",NaN,Kph,2005,3.0,23.0,2005,...,West Bengal,2.944439,8.929435,11.982935,9,24,4,2,28,1
260,Hydrological,Mass movement (wet),Landslide (wet),Kulla (Himachal Pradesh),NaN,NaN,1995,9.0,12.0,1995,...,Himachal Pradesh,5.993961,13.910822,11.982935,8,15,3,0,8,4
206,Hydrological,Flood,Flood (General),Assam province,69190.14,Km2,2015,8.0,13.0,2015,...,Assam,1.791759,9.798183,11.982935,4,8,3,0,2,0
272,Hydrological,Mass movement (wet),Landslide (wet),"Amboori village (Thiruvananthapuram district, ...",NaN,NaN,2001,11.0,9.0,2001,...,Kerala,4.025352,10.308986,11.982935,8,15,3,1,12,4
175,Hydrological,Flood,Riverine flood,"Khammam, West Godavari, Krishna, Guntur distri...",75490.00,Km2,2008,9.0,5.0,2008,...,Andhra Pradesh,4.317488,10.308986,11.982935,4,20,3,0,0,0
91,Meteorological,Extreme temperature,Heat wave,"Andhra Pradesh, Telangana, Maharashtra, Odisha...",48.00,°C,2017,4.0,12.0,2017,...,Odisha,5.579730,10.308986,11.982935,3,13,4,2,19,5
227,Hydrological,Flood,Flood (General),"Lakhimpur, Sontipur, Darrang, Goalpara (Assam ...",NaN,Km2,2020,5.0,24.0,2020,...,Assam,0.693147,10.309053,11.982935,4,8,3,2,2,0
238,Hydrological,Flood,Flood (General),"Himachal Pradesh, Uttarakhand, Delhi, Bihar, M...",NaN,Km2,2023,6.0,25.0,2023,...,Assam,7.333023,16.142788,11.982935,4,8,3,0,2,0


In [ ]:
df_backup=df.copy()

In [13]:
df[['Disaster Type',
    'Disaster Type_encoded']].value_counts().head(10)

Disaster Type                Disaster Type_encoded
Flood                        4                        153
Storm                        9                        107
Mass movement (wet)          8                         43
Extreme temperature          3                         40
Epidemic                     2                         34
Earthquake                   1                         16
Drought                      0                          7
Wildfire                     10                         3
Glacial lake outburst flood  5                          3
Infestation                  6                          1
Name: count, dtype: int64

In [14]:
#separte the target and independent features
features = [
    'Disaster Type_encoded',
    'Disaster Subtype_encoded',
    'Disaster Subgroup_encoded',
    'Season_encoded',
    'Primary_State_encoded',
    'Magnitude Scale_encoded',
    'Start Month',
    'Start Year',
    'Duration'
]

targets = ['Deaths_log', 'Affected_log', 'Damage_log']

x = df[features]
y = df[targets]


In [15]:
#train-test split
x_train_full, x_test, y_train_full, y_test = train_test_split(
    x, y, 
    test_size=0.15,      
    random_state=42      
)

In [ ]:
#train-validation split
x_train, x_val, y_train, y_val = train_test_split(
    x_train_full, y_train_full,
    test_size=0.176,     
    random_state=42
)

In [17]:
print("X_train:", x_train.shape)   
print("X_val:  ", x_val.shape)     
print("X_test: ", x_test.shape)    

print("y_train:", y_train.shape)
print("y_val:  ", y_val.shape)
print("y_test: ", y_test.shape)

print("Total:", x_train.shape[0] + 
            x_val.shape[0] + 
            x_test.shape[0]) 

X_train: (285, 9)
X_val:   (61, 9)
X_test:  (62, 9)
y_train: (285, 3)
y_val:   (61, 3)
y_test:  (62, 3)
Total: 408


In [19]:
#Scaling
scaler = StandardScaler()

x_train_scaled = scaler.fit_transform(x_train)
x_val_scaled   = scaler.transform(x_val)
x_test_scaled  = scaler.transform(x_test)


print(x_train_scaled.mean(axis=0).round(2))
print(x_train_scaled.std(axis=0).round(2))

#save the scaled model
joblib.dump(scaler, r'C:\Users\boser\Desktop\DisasterProject\models\scaler.pkl')

[-0. -0. -0.  0. -0. -0. -0.  0. -0.]
[1. 1. 1. 1. 1. 1. 1. 1. 1.]


['C:\\Users\\boser\\Desktop\\DisasterProject\\models\\scaler.pkl']

In [26]:
models = {
    'Linear Regression':MultiOutputRegressor(
                        LinearRegression()),

    'Ridge Regression' :MultiOutputRegressor(
                        Ridge(alpha=1.0)),

    'Random Forest' : RandomForestRegressor(
                        n_estimators=100,
                        random_state=42),

    'XGBoost' : MultiOutputRegressor(
                        XGBRegressor(
                        n_estimators=100,
                        random_state=42,
                        verbosity=0)),

    'Gradient Boosting' : MultiOutputRegressor(
                        GradientBoostingRegressor(
                        n_estimators=100,
                        random_state=42)),

    'KNN' : MultiOutputRegressor(
                        KNeighborsRegressor(
                        n_neighbors=5))
}

In [ ]:
results=[]
for name, model in models.items():        # level 1

    model.fit(x_train_scaled, y_train)    # level 2

    y_pred = model.predict(x_val_scaled)  # level 2

    for i, target in enumerate(targets):  # level 2

        rmse = root_mean_squared_error(        # level 3
               y_val.iloc[:, i],
               y_pred[:, i],
               )

        mae  = mean_absolute_error(       # level 3
               y_val.iloc[:, i],
               y_pred[:, i])

        r2   = r2_score(                  # level 3
               y_val.iloc[:, i],
               y_pred[:, i])

        results.append({                  # level 3
            'Model'  : name,
            'Target' : target,
            'RMSE'   : round(rmse, 4),
            'MAE'    : round(mae,  4),
            'R²'     : round(r2,   4)
        })
        


[{'Model': 'Linear Regression',
  'Target': 'Deaths_log',
  'RMSE': 1.3859,
  'MAE': 1.0897,
  'R²': 0.1369},
 {'Model': 'Linear Regression',
  'Target': 'Affected_log',
  'RMSE': 2.9361,
  'MAE': 2.3519,
  'R²': 0.0936},
 {'Model': 'Linear Regression',
  'Target': 'Damage_log',
  'RMSE': 1.428,
  'MAE': 0.7776,
  'R²': -0.0399},
 {'Model': 'Ridge Regression',
  'Target': 'Deaths_log',
  'RMSE': 1.3858,
  'MAE': 1.0894,
  'R²': 0.1371},
 {'Model': 'Ridge Regression',
  'Target': 'Affected_log',
  'RMSE': 2.9347,
  'MAE': 2.35,
  'R²': 0.0945},
 {'Model': 'Ridge Regression',
  'Target': 'Damage_log',
  'RMSE': 1.4279,
  'MAE': 0.777,
  'R²': -0.0398},
 {'Model': 'Random Forest',
  'Target': 'Deaths_log',
  'RMSE': 1.4068,
  'MAE': 1.1156,
  'R²': 0.1107},
 {'Model': 'Random Forest',
  'Target': 'Affected_log',
  'RMSE': 2.6714,
  'MAE': 2.0965,
  'R²': 0.2497},
 {'Model': 'Random Forest',
  'Target': 'Damage_log',
  'RMSE': 1.3733,
  'MAE': 0.8183,
  'R²': 0.0382},
 {'Model': 'XGBoost',

In [34]:
results_df=pd.DataFrame(results)
display(results_df)

,Model,Target,RMSE,MAE,R²
0,Linear Regression,Deaths_log,1.3859,1.0897,0.1369
1,Linear Regression,Affected_log,2.9361,2.3519,0.0936
2,Linear Regression,Damage_log,1.4280,0.7776,-0.0399
3,Ridge Regression,Deaths_log,1.3858,1.0894,0.1371
4,Ridge Regression,Affected_log,2.9347,2.3500,0.0945
5,Ridge Regression,Damage_log,1.4279,0.7770,-0.0398
6,Random Forest,Deaths_log,1.4068,1.1156,0.1107
7,Random Forest,Affected_log,2.6714,2.0965,0.2497
8,Random Forest,Damage_log,1.3733,0.8183,0.0382
9,XGBoost,Deaths_log,1.4338,1.1326,0.0762


In [37]:
results_df.sort_values(by=['Target','R²'])

,Model,Target,RMSE,MAE,R²
1,Linear Regression,Affected_log,2.9361,2.3519,0.0936
4,Ridge Regression,Affected_log,2.9347,2.3500,0.0945
16,KNN,Affected_log,2.9253,2.1612,0.1003
13,Gradient Boosting,Affected_log,2.7955,2.1689,0.1784
10,XGBoost,Affected_log,2.7887,2.1716,0.1824
7,Random Forest,Affected_log,2.6714,2.0965,0.2497
17,KNN,Damage_log,1.5933,0.9189,-0.2947
11,XGBoost,Damage_log,1.5512,1.0034,-0.2272
2,Linear Regression,Damage_log,1.4280,0.7776,-0.0399
5,Ridge Regression,Damage_log,1.4279,0.7770,-0.0398


In [39]:
#best models
for target in targets:
    best = results_df[results_df['Target'] == target].sort_values(
           'R²', ascending=False).iloc[0]
    print(f"{target:15} → {best['Model']:20} R²: {best['R²']}")

Deaths_log      → Ridge Regression     R²: 0.1371
Affected_log    → Random Forest        R²: 0.2497
Damage_log      → Random Forest        R²: 0.0382


In [ ]:
for target in targets:
    display(results_df[results_df['Target'] == target].sort_values(
        'R²', ascending=False).iloc[0])

Model     Ridge Regression
Target          Deaths_log
RMSE                1.3858
MAE                 1.0894
R²                  0.1371
Name: 3, dtype: object

Model     Random Forest
Target     Affected_log
RMSE             2.6714
MAE              2.0965
R²               0.2497
Name: 7, dtype: object

Model     Random Forest
Target       Damage_log
RMSE             1.3733
MAE              0.8183
R²               0.0382
Name: 8, dtype: object

In [46]:
#cross-validation

#define k-fold
kf = KFold(n_splits=5, shuffle=True, random_state=42)

cv_models = {
    'Linear Regression' : MultiOutputRegressor(
                          LinearRegression()),
    'Ridge Regression'  : MultiOutputRegressor(
                          Ridge()),
    'Random Forest'     : RandomForestRegressor(
                          n_estimators=100,
                          random_state=42),
    'XGBoost'           : MultiOutputRegressor(
                          XGBRegressor(
                          n_estimators=100,
                          random_state=42,
                          verbosity=0)),
    'Gradient Boosting' : MultiOutputRegressor(
                          GradientBoostingRegressor(
                          n_estimators=100,
                          random_state=42)),
    'KNN'               : MultiOutputRegressor(
                          KNeighborsRegressor(n_neighbors=5))
}

In [48]:
cv_results = []

for name, model in cv_models.items():
    
    scores = cross_val_score(
             model,
             x_train_full,
             y_train_full,
             cv=kf,
             scoring='r2')
    
    cv_results.append({
        'Model'   : name,
        'CV Mean R²' : round(scores.mean(), 4),
        'CV Std R²'  : round(scores.std(),  4),
        'Min R²'     : round(scores.min(),  4),
        'Max R²'     : round(scores.max(),  4)
    })


cv_df = pd.DataFrame(cv_results)
print(cv_df.sort_values(
        by='CV Mean R²', ascending=False))

               Model  CV Mean R²  CV Std R²  Min R²  Max R²
2      Random Forest      0.1224     0.0952 -0.0568  0.2183
1   Ridge Regression      0.0896     0.0400  0.0303  0.1355
0  Linear Regression      0.0895     0.0401  0.0300  0.1358
4  Gradient Boosting      0.0360     0.1248 -0.1774  0.1869
5                KNN     -0.0755     0.0795 -0.2192  0.0039
3            XGBoost     -0.0909     0.1356 -0.2399  0.1238


In [52]:
overfit_results = []

for name, model in models.items():

    model.fit(x_train_scaled, y_train)
    
    y_pred_train = model.predict(x_train_scaled)
    y_pred_val   = model.predict(x_val_scaled)
    
    for i, target in enumerate(['Deaths_log',
                                 'Affected_log',
                                 'Damage_log']):
        
        r2_train = r2_score(y_train.iloc[:,i],
                            y_pred_train[:,i])
        
        r2_val   = r2_score(y_val.iloc[:,i],
                            y_pred_val[:,i])
        
        gap = r2_train - r2_val
        
        if gap > 0.2:
            status = 'Overfitting'
        elif gap > 0.1:
            status = 'Slight Overfit'
        else:
            status = 'Healthy'
        
        overfit_results.append({
            'Model'      : name,
            'Target'     : target,
            'Train R²'   : round(r2_train, 4),
            'Val R²'     : round(r2_val,   4),
            'Gap'        : round(gap,       4),
            'Status'     : status
        })

overfit_df = pd.DataFrame(overfit_results)
display(overfit_df.sort_values(
        by=['Target','Gap'],
        ascending=[True, False]))

,Model,Target,Train R²,Val R²,Gap,Status
10,XGBoost,Affected_log,0.9999,0.1824,0.8175,Overfitting
7,Random Forest,Affected_log,0.8816,0.2497,0.6319,Overfitting
13,Gradient Boosting,Affected_log,0.6618,0.1784,0.4834,Overfitting
16,KNN,Affected_log,0.3927,0.1003,0.2924,Overfitting
1,Linear Regression,Affected_log,0.2091,0.0936,0.1154,Slight Overfit
4,Ridge Regression,Affected_log,0.2091,0.0945,0.1145,Slight Overfit
11,XGBoost,Damage_log,1.0000,-0.2272,1.2272,Overfitting
8,Random Forest,Damage_log,0.8557,0.0382,0.8174,Overfitting
14,Gradient Boosting,Damage_log,0.6679,0.0001,0.6678,Overfitting
17,KNN,Damage_log,0.3025,-0.2947,0.5972,Overfitting


In [55]:
#hyperparameter tuning 
results_tune = []

########################################################
# 1️⃣ Ridge Regression Tuning
########################################################

ridge_param_grid = {
    "alpha":[0.001,0.01,0.1,1,10,100]
}

targets = ["Deaths_log","Affected_log","Damage_log"]

for target in targets:

    ridge = Ridge()

    ridge_grid = GridSearchCV(
        ridge,
        ridge_param_grid,
        cv=5,
        scoring="r2",
        n_jobs=-1
    )

    ridge_grid.fit(x_train, y_train[target])

    best_model = ridge_grid.best_estimator_

    train_pred = best_model.predict(x_train)
    val_pred = best_model.predict(x_val)

    train_r2 = r2_score(y_train[target], train_pred)
    val_r2 = r2_score(y_val[target], val_pred)

    gap = train_r2 - val_r2

    results_tune.append({
        "Model":"Ridge",
        "Target":target,
        "Train_R2":train_r2,
        "Val_R2":val_r2,
        "Gap":gap
    })


########################################################
# 2️⃣ Random Forest Tuning
########################################################

rf_param_grid = {

    "n_estimators":[100,200,300],
    "max_depth":[3,4,5,6,None],
    "min_samples_split":[2,5,10],
    "min_samples_leaf":[1,3,5],
    "max_features":["sqrt","log2"]
}

rf_targets = ["Deaths_log","Affected_log","Damage_log"]

for target in rf_targets:

    rf = RandomForestRegressor(random_state=42)

    rf_grid = GridSearchCV(
        rf,
        rf_param_grid,
        cv=5,
        scoring="r2",
        n_jobs=-1
    )

    rf_grid.fit(x_train, y_train[target])

    best_model = rf_grid.best_estimator_

    train_pred = best_model.predict(x_train)
    val_pred = best_model.predict(x_val)

    train_r2 = r2_score(y_train[target], train_pred)
    val_r2 = r2_score(y_val[target], val_pred)

    gap = train_r2 - val_r2

    results_tune.append({
        "Model":"Random Forest",
        "Target":target,
        "Train_R2":train_r2,
        "Val_R2":val_r2,
        "Gap":gap
    })


########################################################
# 3️⃣ Gradient Boosting Tuning (Deaths only)
########################################################

gb_param_grid = {

    "n_estimators":[100,200,300],
    "learning_rate":[0.01,0.05,0.1],
    "max_depth":[2,3,4],
    "subsample":[0.7,0.8,1]
}

gb = GradientBoostingRegressor(random_state=42)

gb_grid = GridSearchCV(
    gb,
    gb_param_grid,
    cv=5,
    scoring="r2",
    n_jobs=-1
)

gb_grid.fit(x_train, y_train["Deaths_log"])

best_model = gb_grid.best_estimator_

train_pred = best_model.predict(x_train)
val_pred = best_model.predict(x_val)

train_r2 = r2_score(y_train["Deaths_log"], train_pred)
val_r2 = r2_score(y_val["Deaths_log"], val_pred)

gap = train_r2 - val_r2

results_tune.append({
    "Model":"Gradient Boosting",
    "Target":"Deaths_log",
    "Train_R2":train_r2,
    "Val_R2":val_r2,
    "Gap":gap
})


########################################################
# 4️⃣ Create Result Table
########################################################

results_df_tune = pd.DataFrame(results_tune)

results_df_tune = results_df_tune.sort_values(by=["Target","Val_R2"],ascending=False)

print(results_df_tune)

               Model        Target  Train_R2    Val_R2       Gap
3      Random Forest    Deaths_log  0.613015  0.174863  0.438152
6  Gradient Boosting    Deaths_log  0.434937  0.171616  0.263320
0              Ridge    Deaths_log  0.215149  0.139326  0.075823
5      Random Forest    Damage_log  0.284956 -0.000211  0.285167
2              Ridge    Damage_log  0.086110 -0.038569  0.124679
4      Random Forest  Affected_log  0.446916  0.216066  0.230850
1              Ridge  Affected_log  0.203206  0.112726  0.090480


In [56]:
display(results_df_tune)

,Model,Target,Train_R2,Val_R2,Gap
3,Random Forest,Deaths_log,0.613015,0.174863,0.438152
6,Gradient Boosting,Deaths_log,0.434937,0.171616,0.263320
0,Ridge,Deaths_log,0.215149,0.139326,0.075823
5,Random Forest,Damage_log,0.284956,-0.000211,0.285167
2,Ridge,Damage_log,0.086110,-0.038569,0.124679
4,Random Forest,Affected_log,0.446916,0.216066,0.230850
1,Ridge,Affected_log,0.203206,0.112726,0.090480


In [ ]:
d